# EDA — INMET

Esta análise usa as medições horárias das estações disponíveis na Bronze de 2025. O escopo inclui todas as estações coletadas, não apenas as estações das capitais.

**Perguntas:**

1. Quantos registros, arquivos, estações e UFs estão presentes?
2. Há campos vazios, marcadores de ausência ou falhas de conversão?
3. Existem datas inválidas, horários duplicados ou lacunas temporais?
4. Qual a diferença entre cobertura de horários e disponibilidade das medições?
5. Que valores merecem investigação e como as variáveis se comportam em uma estação de exemplo?

A análise não altera a Bronze, não preenche ausências e não seleciona automaticamente estações para representar municípios. Fonte dos arquivos anuais: [INMET — dados históricos](https://portal.inmet.gov.br/dadoshistoricos).


## 1. Bibliotecas, caminhos e execução

Mantemos duas tarefas simultâneas. As verificações abrangem a base inteira; os gráficos recebem apenas agregações ou uma amostra limitada.

A visão numérica reduzida e as chaves horárias usam persistência em disco temporário do Spark, evitando manter milhões de linhas completas na memória. As tabelas persistidas são liberadas ao encerrar a análise.


In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [ ]:
project_directory = Path.cwd().resolve()

while (
    not (project_directory / "pipeline" / "bronze").is_dir()
    or not (project_directory / "data").is_dir()
):

    if project_directory == project_directory.parent:
        raise FileNotFoundError("Execute o notebook a partir de uma pasta dentro do projeto.")

    project_directory = project_directory.parent

YEAR = 2025
INMET_DIRECTORY = project_directory / "data" / "bronze" / "inmet" / str(YEAR)
local_filesystem_jar = (
    project_directory / "pipeline" / "bronze" / "lib" / "hadoop-bare-naked-local-fs-0.1.0.jar"
)

for required_path in [INMET_DIRECTORY, local_filesystem_jar]:

    if not required_path.exists():
        raise FileNotFoundError(f"Caminho necessário não encontrado: {required_path}")

os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder.appName("Viajando com o dado - EDA INMET")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.driver.extraClassPath", str(local_filesystem_jar))
    .config("spark.executor.extraClassPath", str(local_filesystem_jar))
    .config("spark.hadoop.fs.file.impl", "com.globalmentor.apache.hadoop.fs.BareLocalFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"Spark: {spark.version}; ano: {YEAR}; fuso da análise: UTC")
print(f"Entrada: {INMET_DIRECTORY}")


## 2. Princípio estrutural e catálogo das estações

A Bronze mantém medições como texto, metadados da estação e a linha original. A coluna adicional vazia após o último ponto e vírgula não é uma variável meteorológica.

Projetamos apenas identificadores, datas e valores necessários aos cálculos. Cabeçalhos e linhas originais continuam disponíveis na fonte, mas não serão repetidos na visão persistida.


In [ ]:
observations = (
    spark.read.option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.parquet")
    .parquet(INMET_DIRECTORY.resolve().as_posix())
)

observations.printSchema()
observations.select(
    "station_code",
    "station_name",
    "state_code",
    "observation_date",
    "observation_time_utc",
    "air_temperature_c",
    "precipitation_mm",
).show(5, truncate=False)

station_catalog = (
    observations.select(
        "source_file",
        "station_code",
        "station_name",
        "state_code",
        "station_region",
        "station_latitude",
        "station_longitude",
        "station_altitude",
        "station_founded_at",
    )
    .distinct()
    .cache()
)

station_count = station_catalog.select("station_code").distinct().count()
source_file_count = station_catalog.select("source_file").distinct().count()
state_count = station_catalog.select("state_code").distinct().count()

print(f"Estações: {station_count}; arquivos de origem: {source_file_count}; UFs: {state_count}")
station_catalog.orderBy("state_code", "station_code").show(10, truncate=65)
station_catalog.groupBy("state_code").agg(
    F.countDistinct("station_code").alias("station_count")
).orderBy("state_code").show(30, truncate=False)


## 3. Visão numérica e horários UTC

A vírgula decimal é interpretada somente na visão de análise. Cada célula recebe um status:

- `empty`: nulo, vazio ou composto apenas por espaços;
- `sentinel`: valor numérico `-9999`, contado separadamente como marcador de ausência;
- `invalid`: texto não interpretável ou número não finito;
- `valid`: número interpretável, ainda sujeito às verificações de domínio.

Zero continua sendo um valor válido, inclusive para chuva ou vento. Não substituímos ausência de precipitação por zero. Datas e horas são interpretadas em UTC, conforme o cabeçalho `Hora UTC` dos arquivos.

A conversão numérica é materializada em disco antes do cálculo dos status. Essa divisão evita expressões Java excessivamente grandes e repetição das conversões. A tabela intermediária é liberada após preparar a visão final.


In [ ]:
NUMERIC_COLUMNS = [
    "precipitation_mm",
    "station_pressure_mb",
    "station_pressure_max_mb",
    "station_pressure_min_mb",
    "global_radiation_kj_m2",
    "air_temperature_c",
    "dew_point_temperature_c",
    "air_temperature_max_c",
    "air_temperature_min_c",
    "dew_point_temperature_max_c",
    "dew_point_temperature_min_c",
    "relative_humidity_max_percent",
    "relative_humidity_min_percent",
    "relative_humidity_percent",
    "wind_direction_degrees",
    "wind_gust_ms",
    "wind_speed_ms",
]

identifier_columns = [
    "source_file",
    "source_row_number",
    "state_code",
    "station_code",
    "date_raw",
    "time_raw",
]

numeric_columns = [
    F.col("source_file"),
    F.col("source_row_number"),
    F.col("state_code"),
    F.col("station_code"),
    F.col("observation_date").alias("date_raw"),
    F.col("observation_time_utc").alias("time_raw"),
]

# Primeiro interpreta cada número uma única vez e registra se o texto estava vazio.
for column_name in NUMERIC_COLUMNS:

    raw_value = F.col(column_name)
    parsed_value = F.expr(f"try_cast(translate(trim(`{column_name}`), ',', '.') AS DOUBLE)")
    numeric_columns.append(parsed_value.alias(column_name + "_parsed"))
    numeric_columns.append(
        (raw_value.isNull() | (F.trim(raw_value) == "")).alias(column_name + "_empty")
    )

numeric_view = observations.select(*numeric_columns).persist(StorageLevel.DISK_ONLY)
print(f"Linhas interpretadas na primeira etapa: {numeric_view.count():,}")

# A etapa separada evita repetir a conversão nas expressões de status.
selected_columns = []

for column_name in identifier_columns:

    selected_columns.append(F.col(column_name))

for column_name in NUMERIC_COLUMNS:

    parsed_value = F.col(column_name + "_parsed")
    is_empty = F.col(column_name + "_empty")
    is_invalid = (
        parsed_value.isNull() | F.isnan(parsed_value) | (F.abs(parsed_value) == F.lit(float("inf")))
    )

    value_status = (
        F.when(is_empty, "empty")
        .when(parsed_value == -9999, "sentinel")
        .when(is_invalid, "invalid")
        .otherwise("valid")
    )

    selected_columns.append(F.when(value_status == "valid", parsed_value).alias(column_name))
    selected_columns.append(value_status.alias(column_name + "_status"))

analysis_view = numeric_view.select(*selected_columns)
normalized_date = F.regexp_replace(F.trim(F.col("date_raw")), "/", "-")
normalized_time = F.regexp_extract(F.trim(F.col("time_raw")), r"^([0-9]{4}) UTC$", 1)

analysis_view = analysis_view.withColumn(
    "timestamp_utc",
    F.try_to_timestamp(
        F.concat_ws(" ", normalized_date, normalized_time), F.lit("yyyy-MM-dd HHmm")
    ),
).persist(StorageLevel.DISK_ONLY)

record_count = analysis_view.count()
numeric_view.unpersist()
print(f"Medições horárias: {record_count:,}")
analysis_view.select(
    "station_code",
    "date_raw",
    "time_raw",
    "timestamp_utc",
    "air_temperature_c",
    "air_temperature_c_status",
).show(5, truncate=False)


## 4. Ausências e conversões

Os percentuais usam todas as linhas da base como denominador. Vazios e `-9999` são apresentados separadamente dos valores não interpretáveis.

A tabela de coocorrência verifica se temperatura e precipitação ficam indisponíveis juntas. Esse padrão pode orientar a investigação, mas não identifica sozinho a causa da ausência nem seu mecanismo estatístico.


In [ ]:
quality_expressions = []

for column_name in NUMERIC_COLUMNS:

    for status in ["valid", "empty", "sentinel", "invalid"]:

        quality_expressions.append(
            F.sum(F.when(F.col(column_name + "_status") == status, 1).otherwise(0)).alias(
                column_name + "__" + status
            )
        )

quality_totals = analysis_view.agg(*quality_expressions).first()
missing_records = []
missing_cell_count = 0
invalid_numeric_cell_count = 0

for column_name in NUMERIC_COLUMNS:

    valid_count = int(quality_totals[column_name + "__valid"] or 0)
    empty_count = int(quality_totals[column_name + "__empty"] or 0)
    sentinel_count = int(quality_totals[column_name + "__sentinel"] or 0)
    invalid_count = int(quality_totals[column_name + "__invalid"] or 0)
    missing_percent = 0.0

    if record_count > 0:
        missing_percent = round(100.0 * (empty_count + sentinel_count) / record_count, 2)

    missing_records.append(
        (column_name, valid_count, empty_count, sentinel_count, invalid_count, missing_percent)
    )
    missing_cell_count += empty_count + sentinel_count
    invalid_numeric_cell_count += invalid_count

missing_summary = spark.createDataFrame(
    missing_records,
    "variable string, valid_count long, empty_count long, sentinel_count long, invalid_count long, missing_percent double",
)

missing_summary.orderBy(F.desc("missing_percent")).show(20, truncate=False)

analysis_view.groupBy(
    F.col("air_temperature_c").isNotNull().alias("temperature_available"),
    F.col("precipitation_mm").isNotNull().alias("precipitation_available"),
).count().orderBy("temperature_available", "precipitation_available").show(truncate=False)


## 5. Metadados e UFs

Verificamos identificadores ausentes, siglas fora do conjunto de 27 UFs, coordenadas fora dos limites globais e códigos de estação associados a várias localizações ou UFs.

Múltiplas localizações podem exigir análise de histórico da estação, não remoção automática. Coordenadas dentro dos limites globais não comprovam localização dentro da UF.


In [ ]:
EXPECTED_STATES = [
    "AC",
    "AL",
    "AM",
    "AP",
    "BA",
    "CE",
    "DF",
    "ES",
    "GO",
    "MA",
    "MG",
    "MS",
    "MT",
    "PA",
    "PB",
    "PE",
    "PI",
    "PR",
    "RJ",
    "RN",
    "RO",
    "RR",
    "RS",
    "SC",
    "SE",
    "SP",
    "TO",
]

missing_identifier = (
    F.col("station_code").isNull()
    | (F.trim(F.col("station_code")) == "")
    | F.col("state_code").isNull()
    | (F.trim(F.col("state_code")) == "")
)
missing_identifier_count = analysis_view.filter(missing_identifier).count()
unexpected_states = station_catalog.filter(
    F.col("state_code").isNull() | ~F.col("state_code").isin(EXPECTED_STATES)
)
unexpected_state_count = unexpected_states.select("state_code").distinct().count()

station_locations = station_catalog.withColumn(
    "latitude", F.expr("try_cast(regexp_replace(trim(station_latitude), ',', '.') AS DOUBLE)")
).withColumn(
    "longitude", F.expr("try_cast(regexp_replace(trim(station_longitude), ',', '.') AS DOUBLE)")
)

location_valid = F.coalesce(
    F.col("latitude").between(-90, 90)
    & ~F.isnan("latitude")
    & F.col("longitude").between(-180, 180)
    & ~F.isnan("longitude"),
    F.lit(False),
)

invalid_locations = station_locations.filter(~location_valid)
invalid_location_count = invalid_locations.count()

metadata_conflicts = (
    station_catalog.groupBy("station_code")
    .agg(
        F.countDistinct("state_code").alias("state_variants"),
        F.countDistinct(F.struct("station_latitude", "station_longitude")).alias(
            "location_variants"
        ),
    )
    .filter((F.col("state_variants") > 1) | (F.col("location_variants") > 1))
)

metadata_conflict_count = metadata_conflicts.count()

print(f"Registros sem identificadores: {missing_identifier_count}")
print(
    f"Siglas inesperadas: {unexpected_state_count}; metadados com coordenadas inválidas: {invalid_location_count}"
)

print(f"Estações com metadados divergentes: {metadata_conflict_count}")
unexpected_states.select("station_code", "state_code").show(truncate=False)
invalid_locations.select("station_code", "station_name", "latitude", "longitude").show(
    truncate=False
)
metadata_conflicts.show(truncate=False)


## 6. Horários, duplicações e cobertura temporal

A chave investigada é estação + horário UTC. A cobertura considera horários interpretáveis, alinhados à hora cheia e pertencentes ao ano selecionado.

Mostramos a cobertura do ano completo e a do intervalo entre o primeiro e o último horário observado. A segunda não exige registros anteriores ao início da série disponível; lacunas antes ou depois desse intervalo precisam de contexto externo.

Ter uma linha para uma hora não significa ter todas as variáveis preenchidas nessa hora.


In [ ]:
invalid_timestamp_count = analysis_view.filter(F.col("timestamp_utc").isNull()).count()
outside_year_count = analysis_view.filter(
    F.col("timestamp_utc").isNotNull() & (F.year("timestamp_utc") != YEAR)
).count()
nonhourly_count = analysis_view.filter(
    F.col("timestamp_utc").isNotNull()
    & ((F.minute("timestamp_utc") != 0) | (F.second("timestamp_utc") != 0))
).count()

hourly_rows = analysis_view.filter(
    F.col("timestamp_utc").isNotNull()
    & (F.year("timestamp_utc") == YEAR)
    & (F.minute("timestamp_utc") == 0)
    & (F.second("timestamp_utc") == 0)
    & F.col("station_code").isNotNull()
    & (F.trim(F.col("station_code")) != "")
)

hour_counts = (
    hourly_rows.groupBy("station_code", "timestamp_utc").count().persist(StorageLevel.DISK_ONLY)
)

station_coverage = hour_counts.groupBy("station_code").agg(
    F.count("*").alias("observed_hours"),
    F.sum("count").alias("observation_rows"),
    F.min("timestamp_utc").alias("first_timestamp_utc"),
    F.max("timestamp_utc").alias("last_timestamp_utc"),
    F.sum(F.when(F.col("count") > 1, F.col("count") - 1).otherwise(0)).alias(
        "duplicate_extra_rows"
    ),
)

annual_hours = int((datetime(YEAR + 1, 1, 1) - datetime(YEAR, 1, 1)).total_seconds() / 3600)

station_coverage = (
    station_coverage.withColumn(
        "interval_hours",
        (
            (F.unix_timestamp("last_timestamp_utc") - F.unix_timestamp("first_timestamp_utc"))
            / 3600
            + 1
        ).cast("long"),
    )
    .withColumn("missing_inside_interval", F.col("interval_hours") - F.col("observed_hours"))
    .withColumn(
        "annual_coverage_percent", F.round(100.0 * F.col("observed_hours") / F.lit(annual_hours), 2)
    )
    .cache()
)

coverage_totals = station_coverage.agg(
    F.sum("duplicate_extra_rows").alias("duplicate_extra_rows"),
    F.sum("missing_inside_interval").alias("missing_inside_interval"),
).first()

duplicate_extra_count = int(coverage_totals["duplicate_extra_rows"] or 0)
missing_interval_hour_count = int(coverage_totals["missing_inside_interval"] or 0)
partial_year_station_count = station_coverage.filter(F.col("observed_hours") < annual_hours).count()

print(f"Datas/horas não interpretadas: {invalid_timestamp_count}")
print(f"Registros fora do ano: {outside_year_count}; fora da hora cheia: {nonhourly_count}")
print(f"Registros extras com chave estação/horário repetida: {duplicate_extra_count}")
print(f"Horas ausentes dentro dos intervalos observados: {missing_interval_hour_count}")
print(f"Estações com menos de {annual_hours} horas no ano: {partial_year_station_count}")

hour_counts.filter(F.col("count") > 1).orderBy(F.desc("count")).show(15, truncate=False)
station_coverage.orderBy("annual_coverage_percent", "station_code").show(20, truncate=False)


## 7. Disponibilidade das medições por estação

Os percentuais de temperatura, chuva e vento referem-se às linhas disponíveis de cada estação, não ao ano inteiro. Eles complementam a cobertura temporal e incluem como indisponíveis os vazios, marcadores e valores não interpretáveis.


In [ ]:
station_quality = analysis_view.groupBy("station_code").agg(
    F.count("*").alias("record_count"),
    F.count("air_temperature_c").alias("temperature_count"),
    F.count("precipitation_mm").alias("precipitation_count"),
    F.count("wind_speed_ms").alias("wind_count"),
)

for variable in ["temperature", "precipitation", "wind"]:

    station_quality = station_quality.withColumn(
        variable + "_available_percent",
        F.round(100.0 * F.col(variable + "_count") / F.col("record_count"), 2),
    )

station_quality = station_quality.cache()

station_labels = station_catalog.groupBy("station_code").agg(
    F.sort_array(F.collect_set("station_name")).alias("station_names"),
    F.sort_array(F.collect_set("state_code")).alias("states"),
)

station_quality.join(station_labels, "station_code", "left").orderBy(
    "temperature_available_percent", "station_code"
).show(20, truncate=50)


## 8. Regras de domínio e consistência

Sinalizamos chuva, radiação e velocidades negativas; umidade fora de 0–100%; direção fora de 0–360 graus; pressão não positiva; e máximas menores que mínimas no mesmo intervalo.

Não exigimos temperatura positiva. Também não comparamos automaticamente a leitura instantânea com extremos da hora anterior, pois as janelas de referência são diferentes. Os alertas orientam investigação; não corrigem os dados.


In [ ]:
domain_rules = [
    ("negative_precipitation", F.col("precipitation_mm") < 0),
    ("negative_radiation", F.col("global_radiation_kj_m2") < 0),
    ("negative_wind_speed", F.col("wind_speed_ms") < 0),
    ("negative_wind_gust", F.col("wind_gust_ms") < 0),
    ("wind_direction_outside_range", ~F.col("wind_direction_degrees").between(0, 360)),
    ("humidity_outside_range", ~F.col("relative_humidity_percent").between(0, 100)),
    ("humidity_max_outside_range", ~F.col("relative_humidity_max_percent").between(0, 100)),
    ("humidity_min_outside_range", ~F.col("relative_humidity_min_percent").between(0, 100)),
    ("nonpositive_pressure", F.col("station_pressure_mb") <= 0),
    ("nonpositive_pressure_max", F.col("station_pressure_max_mb") <= 0),
    ("nonpositive_pressure_min", F.col("station_pressure_min_mb") <= 0),
    ("temperature_max_below_min", F.col("air_temperature_max_c") < F.col("air_temperature_min_c")),
    (
        "dewpoint_max_below_min",
        F.col("dew_point_temperature_max_c") < F.col("dew_point_temperature_min_c"),
    ),
    ("pressure_max_below_min", F.col("station_pressure_max_mb") < F.col("station_pressure_min_mb")),
    (
        "humidity_max_below_min",
        F.col("relative_humidity_max_percent") < F.col("relative_humidity_min_percent"),
    ),
]

domain_expressions = []
any_domain_issue = F.lit(False)

for rule_name, condition in domain_rules:

    domain_expressions.append(F.sum(F.when(condition, 1).otherwise(0)).alias(rule_name))
    any_domain_issue = any_domain_issue | F.coalesce(condition, F.lit(False))

domain_totals = analysis_view.agg(*domain_expressions).first()
domain_records = []

for rule_name, condition in domain_rules:

    domain_records.append((rule_name, int(domain_totals[rule_name] or 0)))

domain_summary = spark.createDataFrame(domain_records, "rule string, flagged_count long")
domain_summary.orderBy(F.desc("flagged_count"), "rule").show(20, truncate=False)
domain_flagged_row_count = analysis_view.filter(any_domain_issue).count()

print(f"Linhas com pelo menos um alerta de domínio: {domain_flagged_row_count}")

analysis_view.filter(any_domain_issue).select(
    "station_code",
    "state_code",
    "timestamp_utc",
    "precipitation_mm",
    "air_temperature_c",
    "relative_humidity_percent",
    "global_radiation_kj_m2",
    "wind_speed_ms",
).orderBy("station_code", "timestamp_utc").show(15, truncate=False)


## 9. Estatística descritiva

Os resumos usam números interpretáveis, sem vazios ou `-9999`, mas ainda podem conter valores sinalizados pelas regras de domínio. A tabela permite identificar extremos antes de decidir os tratamentos.

As estatísticas nacionais descrevem os registros disponíveis: estações com mais horas têm maior peso. Elas não representam uma média territorial do clima do Brasil.


In [ ]:
PRIMARY_VARIABLES = [
    "air_temperature_c",
    "precipitation_mm",
    "relative_humidity_percent",
    "wind_speed_ms",
    "global_radiation_kj_m2",
]

analysis_view.select(*PRIMARY_VARIABLES).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show(truncate=False)

rain_counts = analysis_view.agg(
    F.count("precipitation_mm").alias("valid_precipitation_count"),
    F.sum(F.when(F.col("precipitation_mm") == 0, 1).otherwise(0)).alias("zero_precipitation_count"),
).first()

zero_precipitation_percent = 0.0

if rain_counts["valid_precipitation_count"] > 0:
    zero_precipitation_percent = (
        100.0 * rain_counts["zero_precipitation_count"] / rain_counts["valid_precipitation_count"]
    )

print(
    f"Horas com chuva igual a zero entre valores interpretáveis: {zero_precipitation_percent:.2f}%"
)


## 10. Visualização das ausências

A barra considera vazios, marcadores e falhas de interpretação separadamente. Ausência de radiação à noite, por exemplo, não é transformada em zero sem investigar a convenção da fonte.


In [ ]:
missing_plot_rows = missing_summary.orderBy(F.desc("missing_percent"), "variable").collect()
variable_labels = []
empty_percentages = []
sentinel_percentages = []
invalid_percentages = []

for row in missing_plot_rows:

    denominator = max(record_count, 1)
    variable_labels.append(row["variable"])
    empty_percentages.append(100.0 * row["empty_count"] / denominator)
    sentinel_percentages.append(100.0 * row["sentinel_count"] / denominator)
    invalid_percentages.append(100.0 * row["invalid_count"] / denominator)

sentinel_left = []
invalid_left = []

for empty_percent, sentinel_percent in zip(empty_percentages, sentinel_percentages):

    sentinel_left.append(empty_percent)
    invalid_left.append(empty_percent + sentinel_percent)

figure, axis = plt.subplots(figsize=(12, 7))
axis.barh(variable_labels, empty_percentages, label="Vazio/null", color="#2d223a")
axis.barh(
    variable_labels,
    sentinel_percentages,
    left=sentinel_left,
    label="Marcador -9999",
    color="#b68c82",
)
axis.barh(
    variable_labels,
    invalid_percentages,
    left=invalid_left,
    label="Não interpretável",
    color="tab:red",
)
axis.invert_yaxis()
axis.set_xlim(0, 100)
axis.set_xlabel("Percentual dos registros (%)")
axis.set_title("Disponibilidade das variáveis — INMET 2025")
axis.legend()

figure.tight_layout()
plt.show()


## 11. Distribuições em amostra limitada

As verificações anteriores usam todos os registros. Para os histogramas, usamos uma amostra com semente fixa, limitada a 20 mil linhas. O tamanho efetivo é informado.

A chuva é mostrada apenas nas horas com valor positivo e o vento com valores não negativos; esses recortes são visuais. Temperaturas de estações e épocas do ano diferentes ficam misturadas nesta visão geral, por isso a distribuição não deve ser usada para impor normalidade ou remover extremos automaticamente.


In [ ]:
sample_fraction = min(1.0, 20000.0 / max(record_count, 1))

sample_rows = (
    analysis_view.select("air_temperature_c", "precipitation_mm", "wind_speed_ms")
    .sample(withReplacement=False, fraction=sample_fraction, seed=42)
    .limit(20000)
    .collect()
)

temperature_values = []
positive_rain_values = []
wind_values = []

for row in sample_rows:

    if row["air_temperature_c"] is not None:
        temperature_values.append(row["air_temperature_c"])

    if row["precipitation_mm"] is not None and row["precipitation_mm"] > 0:
        positive_rain_values.append(row["precipitation_mm"])

    if row["wind_speed_ms"] is not None and row["wind_speed_ms"] >= 0:
        wind_values.append(row["wind_speed_ms"])

print(f"Linhas na amostra: {len(sample_rows)}")

figure, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_values = [temperature_values, positive_rain_values, wind_values]
plot_titles = ["Temperatura do ar", "Chuva nas horas com valor > 0", "Velocidade do vento >= 0"]
plot_units = ["Temperatura (°C)", "Precipitação horária (mm)", "Velocidade (m/s)"]

for axis, values, title, unit in zip(axes, plot_values, plot_titles, plot_units):

    axis.hist(values, bins=35, color="#2d223a", edgecolor="white")
    axis.set_title(title)
    axis.set_xlabel(unit)
    axis.set_ylabel("Frequência na amostra")

figure.tight_layout()
plt.show()


## 12. Evolução diária de uma estação

Usamos `A701` (São Paulo — Mirante) quando disponível; caso contrário, selecionamos uma estação com alta disponibilidade de temperatura. Este exemplo não representa todas as estações nem todo o município.

A temperatura diária é a média das horas disponíveis, com sua contagem apresentada na tabela. Para chuva, mostramos o total diário somente quando há 24 valores interpretáveis, horários distintos e nenhum alerta de chuva negativa. Dias incompletos ficam como lacunas, não como chuva zero. As datas continuam em UTC; uma comparação com dias locais da Open-Meteo exige alinhar os fusos.


In [ ]:
example_station_code = "A701"

if station_quality.filter(F.col("station_code") == example_station_code).count() == 0:
    example_station_code = station_quality.orderBy(
        F.desc("temperature_count"), "station_code"
    ).first()["station_code"]

example_observations = hourly_rows.filter(F.col("station_code") == example_station_code)
example_duplicate_count = hour_counts.filter(
    (F.col("station_code") == example_station_code) & (F.col("count") > 1)
).count()

example_daily = (
    example_observations.groupBy(F.to_date("timestamp_utc").alias("date_utc"))
    .agg(
        F.avg("air_temperature_c").alias("temperature_mean_c"),
        F.count("air_temperature_c").alias("temperature_hours"),
        F.count("precipitation_mm").alias("precipitation_hours"),
        F.countDistinct("timestamp_utc").alias("distinct_hours"),
        F.count("*").alias("record_count"),
        F.sum("precipitation_mm").alias("precipitation_sum_mm"),
        F.sum(F.when(F.col("precipitation_mm") < 0, 1).otherwise(0)).alias("negative_rain_hours"),
    )
    .withColumn(
        "complete_precipitation_mm",
        F.when(
            (F.col("precipitation_hours") == 24)
            & (F.col("distinct_hours") == 24)
            & (F.col("record_count") == 24)
            & (F.col("negative_rain_hours") == 0),
            F.col("precipitation_sum_mm"),
        ),
    )
    .cache()
)

print(
    f"Estação de exemplo: {example_station_code}; chaves horárias duplicadas: {example_duplicate_count}"
)
station_catalog.filter(F.col("station_code") == example_station_code).select(
    "station_code", "station_name", "state_code"
).show(truncate=False)
example_daily.orderBy("precipitation_hours", "date_utc").show(15, truncate=False)

daily_rows = example_daily.orderBy("date_utc").collect()
dates = []
daily_temperatures = []
daily_rain = []

for row in daily_rows:

    dates.append(row["date_utc"])

    if row["temperature_mean_c"] is None:
        daily_temperatures.append(float("nan"))
    else:
        daily_temperatures.append(row["temperature_mean_c"])

    if row["complete_precipitation_mm"] is None:
        daily_rain.append(float("nan"))
    else:
        daily_rain.append(row["complete_precipitation_mm"])

figure, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(dates, daily_temperatures, color="#2d223a", linewidth=1)
axes[0].set_title(f"Estação {example_station_code} — temperatura e chuva diária em UTC")
axes[0].set_ylabel("Temperatura média (°C)")
axes[1].bar(dates, daily_rain, color="#2d223a", width=1)
axes[1].set_ylabel("Chuva em dias completos (mm)")
axes[1].set_xlabel("Data UTC")
figure.autofmt_xdate()
figure.tight_layout()
plt.show()


## 13. IQR com contexto temporal

Aplicamos a regra do IQR à temperatura da estação de exemplo separadamente por mês. Isso reduz a mistura entre estações do ano; ainda assim, ondas de calor ou frio podem gerar extremos legítimos.

Os quartis são estimados pelo Spark. Nenhuma observação é removida, e as contagens são sinais para investigação, não uma conclusão automática de erro.


In [ ]:
example_temperature = example_observations.filter(
    F.col("air_temperature_c").isNotNull()
).withColumn("month", F.month("timestamp_utc"))

monthly_quartiles = (
    example_temperature.groupBy("month")
    .agg(F.percentile_approx("air_temperature_c", [0.25, 0.75], 10000).alias("quartiles"))
    .withColumn("q1", F.get("quartiles", 0))
    .withColumn("q3", F.get("quartiles", 1))
)

example_with_bounds = example_temperature.join(monthly_quartiles, "month", "left")
example_with_bounds = example_with_bounds.withColumn(
    "lower_bound", F.col("q1") - 1.5 * (F.col("q3") - F.col("q1"))
)
example_with_bounds = example_with_bounds.withColumn(
    "upper_bound", F.col("q3") + 1.5 * (F.col("q3") - F.col("q1"))
)
example_with_bounds = example_with_bounds.withColumn(
    "iqr_flag",
    (F.col("air_temperature_c") < F.col("lower_bound"))
    | (F.col("air_temperature_c") > F.col("upper_bound")),
)

monthly_iqr = (
    example_with_bounds.groupBy("month", "q1", "q3", "lower_bound", "upper_bound")
    .agg(
        F.count("*").alias("valid_temperature_hours"),
        F.sum(F.when(F.col("iqr_flag"), 1).otherwise(0)).alias("flagged_hours"),
    )
    .orderBy("month")
)

monthly_iqr.show(12, truncate=False)
example_iqr_count = example_with_bounds.filter(F.col("iqr_flag")).count()
print(f"Horas sinalizadas por IQR mensal na estação {example_station_code}: {example_iqr_count}")


## 14. Síntese e propostas para a Silver

As contagens de células não equivalem a linhas: uma observação pode ter várias variáveis ausentes. Cobertura anual parcial e ausência de medições são apresentadas como contexto, sem presumir que a estação deveria operar durante todo o ano.

**Encaminhamentos:** formalizar interpretação de vírgulas e marcadores; manter UTC e definir a conversão para dias locais quando necessária; investigar duplicações e alertas de domínio; definir critérios mínimos de cobertura por variável antes de agregações; escolher explicitamente quais estações representam o recorte das capitais. Esta EDA não faz imputação nem corrige extremos automaticamente.


In [ ]:
print(f"Registros analisados: {record_count:,}")
print(f"Estações: {station_count}; arquivos: {source_file_count}; UFs: {state_count}")
print(f"Células vazias ou com marcador -9999 nas 17 variáveis: {missing_cell_count:,}")
print(f"Células não interpretáveis: {invalid_numeric_cell_count:,}")
print(f"Registros sem identificadores: {missing_identifier_count}")
print(
    f"Datas/horas inválidas: {invalid_timestamp_count}; fora do ano: {outside_year_count}; fora da hora cheia: {nonhourly_count}"
)
print(f"Registros extras por duplicação de estação/horário: {duplicate_extra_count}")
print(f"Horas ausentes dentro dos intervalos observados: {missing_interval_hour_count}")
print(f"Estações com cobertura parcial do ano: {partial_year_station_count}")
print(
    f"UFs inesperadas: {unexpected_state_count}; coordenadas inválidas nos metadados: {invalid_location_count}"
)
print(f"Estações com metadados divergentes: {metadata_conflict_count}")
print(f"Linhas sinalizadas pelas regras de domínio: {domain_flagged_row_count}")
print(
    f"Extremos de temperatura por IQR mensal na estação de exemplo, mantidos: {example_iqr_count}"
)
print("Os alertas identificam pontos para investigação e definição das regras da Silver.")
print("A Bronze foi apenas lida; nenhum registro foi corrigido, removido ou preenchido.")


## 15. Encerramento

Liberamos as tabelas persistidas e encerramos o Spark. Para voltar a usar os DataFrames, execute novamente o notebook desde o início.


In [ ]:
example_daily.unpersist()
station_quality.unpersist()
station_coverage.unpersist()
hour_counts.unpersist()
analysis_view.unpersist()
station_catalog.unpersist()
spark.stop()
